# UK Carbon Intensity — Real-Time Grid Analytics

**Author:** Jai Sooryah Gunaseelan  ·  **Stack:** Python, requests, pandas, seaborn

This notebook pulls **live** data from the National Grid ESO **Carbon Intensity API**
(no API key required) and analyses the UK electricity grid: today's intensity profile,
forecast accuracy, and the current generation mix. The same logic powers a live
Streamlit dashboard (`app.py`).

> When run online it fetches **live** data; offline it falls back to bundled real
> sample responses so the notebook always runs.


## 1. Fetch live grid data

In [ ]:
import sys, pathlib
# add the repo root (parent of this notebook's folder) to the path
root = pathlib.Path.cwd()
root = root.parent if root.name == "notebooks" else root
sys.path.insert(0, str(root))

import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import api, transform, insights, config
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (11, 4)

today = transform.intensity_series(api.intensity_today())
gen   = transform.generation_frame(api.generation_mix())
print(f"Loaded {len(today)} half-hourly periods and {len(gen)} generation fuels")
today.head()

## 2. Today's carbon intensity: forecast vs actual

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(today["time"], today["forecast"], "--", label="Forecast")
ax.plot(today["time"], today["actual"], label="Actual", linewidth=2)
ax.set_ylabel("gCO2/kWh"); ax.set_title("UK grid carbon intensity today"); ax.legend()
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/"intensity_today.png", dpi=110, bbox_inches="tight"); plt.show()

s = insights.day_summary(today)
gw, dw = insights.greenest_window(today), insights.dirtiest_window(today)
print(f"Average: {s['avg']} gCO2/kWh  (min {s['min']}, max {s['max']})")
print(f"Greenest half-hour: {gw['time']:%H:%M} ({gw['intensity']} gCO2/kWh)")
print(f"Dirtiest half-hour: {dw['time']:%H:%M} ({dw['intensity']} gCO2/kWh)")

Intensity typically dips midday (solar) and peaks in the evening as demand rises and the grid leans on gas.

## 3. How accurate is the published forecast?

In [ ]:
acc = insights.forecast_accuracy(today)
print(f"Mean absolute error: {acc['mae']} gCO2/kWh across {acc['n']} settled half-hours")
d = today.dropna(subset=["actual"]).copy()
d["error"] = d["forecast"] - d["actual"]
sns.histplot(d["error"], kde=True, color="slateblue")
plt.title("Forecast error (forecast - actual)"); plt.xlabel("gCO2/kWh")
plt.tight_layout(); plt.show()

## 4. Current generation mix

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,4.5))
sns.barplot(data=gen, x="perc", y="fuel", hue="category", dodge=False, ax=ax[0])
ax[0].set_title("Generation mix by fuel"); ax[0].set_xlabel("% of generation"); ax[0].set_ylabel("")
cat = gen.groupby("category")["perc"].sum().sort_values(ascending=False)
ax[1].pie(cat, labels=cat.index, autopct="%1.0f%%", colors=["#2ecc71","#3498db","#e74c3c","#95a5a6"])
ax[1].set_title("By category")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/"generation_mix.png", dpi=110, bbox_inches="tight"); plt.show()
print(f"Renewables: {insights.renewable_share(gen):.0f}%  |  Low-carbon (incl. nuclear): {insights.low_carbon_share(gen):.0f}%")

## 5. Takeaways

- The grid's carbon intensity swings widely across a single day, driven mostly by **wind and solar** availability vs **gas**.
- The **greenest window** is the practical output: shifting flexible demand (EV charging, appliances, industrial loads) into it cuts real emissions.
- The published **forecast** tracks actuals closely (low MAE), so it's reliable for planning ahead.

*Live source: National Grid ESO Carbon Intensity API. Re-run any time for fresh numbers.*
